# Resume NER Training - FROM SCRATCH (No Pre-trained Models)

This notebook trains a BiLSTM-CRF model completely from scratch.

**Key Difference from BERT approach:**
- ❌ No pre-trained models (BERT, etc.)
- ✅ Random weight initialization
- ✅ Builds vocabulary from your data
- ✅ Trains everything from scratch

## Setup Steps:
1. Enable GPU: Runtime → Change runtime type → GPU
2. Upload your dataset to Google Drive:
   - **RECOMMENDED**: `dataturks_resumes_fixed.json` (200 examples, excellent quality)
   - **Alternative**: `train_aggressive_cleaned.json` (5,943 examples, larger dataset)
3. Mount Google Drive and update the dataset path in Step 2
4. Run all cells below

## Advanced Optimizations Applied:
- ✅ Using high-quality dataset (DataTurks - email issues fixed)
- ✅ **Advanced architecture**: 200 embedding dim, 512 hidden dim, 4 LSTM layers
- ✅ **Character-level embeddings** (multi-filter CNN for OOV words)
- ✅ **Multi-head attention** (8 heads, residual connections, layer norm)
- ✅ **Feed-forward network** with residual connections and GELU activation
- ✅ **Layer normalization** throughout (better training stability)
- ✅ **Class weighting** (handles imbalanced data)
- ✅ **Learning rate warmup** (5 epochs) + scheduling
- ✅ **Weight decay** (L2 regularization)
- ✅ **Gradient clipping** (prevents exploding gradients)
- ✅ More epochs (60) with early stopping (patience=12)
- ✅ Optimized learning rate (0.0008) for stability
- ✅ Fixed loss calculation (proper attention masks)

## Step 1: Install Dependencies

In [1]:
!pip install torch numpy tqdm

## Step 2: Load Dataset from Google Drive

In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted!")

# Update the path below to match where your dataset is stored in Drive
# OPTION 1: DataTurks dataset (RECOMMENDED - better quality, 200 examples)
DRIVE_DATASET_PATH = '/content/drive/MyDrive/DATASETS/dataturks_resumes_fixed.json'  # ← RECOMMENDED

# OPTION 2: Your cleaned dataset (larger, 5,943 examples)
# DRIVE_DATASET_PATH = '/content/drive/MyDrive/DATASETS/dataset-5000/train_aggressive_cleaned.json'

print(f"📁 Dataset location: {DRIVE_DATASET_PATH}")
print("✅ Using DataTurks dataset (email issues fixed, excellent quality)")

: 

## Step 3: Create Model Architecture (FROM SCRATCH)

In [ ]:
%%writefile model_from_scratch.py
"""
ADVANCED BiLSTM-CRF model architecture for NER - optimized for high accuracy.
Features: Character embeddings, Multi-head attention, Layer normalization, Residual connections.
No pre-trained models or weights used.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class CharCNN(nn.Module):
    """Character-level CNN for better OOV word handling."""
    def __init__(self, char_vocab_size=128, char_embed_dim=30, char_hidden_dim=50):
        super(CharCNN, self).__init__()
        self.char_embedding = nn.Embedding(char_vocab_size, char_embed_dim, padding_idx=0)
        # Multiple filter sizes for richer character representations
        self.conv1 = nn.Conv1d(char_embed_dim, char_hidden_dim, kernel_size=2, padding=1)
        self.conv2 = nn.Conv1d(char_embed_dim, char_hidden_dim, kernel_size=3, padding=1)
        self.conv3 = nn.Conv1d(char_embed_dim, char_hidden_dim, kernel_size=4, padding=2)
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Linear(char_hidden_dim * 3, char_hidden_dim)
        
    def forward(self, char_ids):
        # char_ids: [batch_size, seq_len, word_len]
        batch_size, seq_len, word_len = char_ids.shape
        char_ids = char_ids.view(-1, word_len)  # [batch*seq, word_len]
        
        char_emb = self.char_embedding(char_ids)  # [batch*seq, word_len, char_embed_dim]
        char_emb = char_emb.transpose(1, 2)  # [batch*seq, char_embed_dim, word_len]
        
        # Multiple filter sizes
        conv1_out = F.relu(self.conv1(char_emb))  # [batch*seq, char_hidden_dim, word_len]
        conv2_out = F.relu(self.conv2(char_emb))
        conv3_out = F.relu(self.conv3(char_emb))
        
        # Pooling
        pooled1 = self.pool(conv1_out).squeeze(-1)  # [batch*seq, char_hidden_dim]
        pooled2 = self.pool(conv2_out).squeeze(-1)
        pooled3 = self.pool(conv3_out).squeeze(-1)
        
        # Concatenate and project
        combined = torch.cat([pooled1, pooled2, pooled3], dim=-1)  # [batch*seq, char_hidden_dim*3]
        char_repr = self.fc(combined)  # [batch*seq, char_hidden_dim]
        
        return char_repr.view(batch_size, seq_len, -1)  # [batch_size, seq_len, char_hidden_dim]


class MultiHeadAttention(nn.Module):
    """Multi-head self-attention with residual connection and layer norm."""
    def __init__(self, hidden_dim, num_heads=8, dropout=0.1):
        super(MultiHeadAttention, self).__init__()
        assert hidden_dim % num_heads == 0
        
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        
        self.q_linear = nn.Linear(hidden_dim, hidden_dim)
        self.k_linear = nn.Linear(hidden_dim, hidden_dim)
        self.v_linear = nn.Linear(hidden_dim, hidden_dim)
        self.out_linear = nn.Linear(hidden_dim, hidden_dim)
        
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(hidden_dim)
        
    def forward(self, x, mask=None):
        # x: [batch_size, seq_len, hidden_dim]
        residual = x
        x = self.layer_norm(x)
        
        batch_size, seq_len, _ = x.shape
        
        # Linear projections
        Q = self.q_linear(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_linear(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_linear(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)
        
        if mask is not None:
            mask = mask.unsqueeze(1).unsqueeze(1)  # [batch, 1, 1, seq_len]
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        attn_output = torch.matmul(attn_weights, V)  # [batch, num_heads, seq_len, head_dim]
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.hidden_dim)
        
        output = self.out_linear(attn_output)
        output = self.dropout(output)
        
        return self.layer_norm(residual + output)  # Residual connection


class BiLSTM_CRF_NER(nn.Module):
    """
    ADVANCED BiLSTM-CRF for Named Entity Recognition.
    Features: Word + Character embeddings, Multi-head attention, Layer normalization, Residual connections.
    All weights initialized randomly - no pre-training.
    """
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_labels, 
                 num_layers=3, dropout=0.5, use_char_emb=True, use_attention=True):
        super(BiLSTM_CRF_NER, self).__init__()
        
        self.use_char_emb = use_char_emb
        self.use_attention = use_attention
        
        # Word embeddings (random initialization - FROM SCRATCH)
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # Character-level embeddings (for better OOV handling)
        if use_char_emb:
            self.char_cnn = CharCNN(char_vocab_size=128, char_embed_dim=30, char_hidden_dim=50)
            input_dim = embedding_dim + 50  # Word + Char embeddings
        else:
            input_dim = embedding_dim
        
        # Layer normalization for embeddings
        self.embed_norm = nn.LayerNorm(input_dim)
        
        # Bidirectional LSTM (larger capacity)
        self.lstm = nn.LSTM(
            input_dim, 
            hidden_dim, 
            num_layers=num_layers,
            bidirectional=True, 
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        # Multi-head attention
        if use_attention:
            self.attention = MultiHeadAttention(hidden_dim * 2, num_heads=8, dropout=dropout)
        
        # Dropout for regularization
        self.dropout = nn.Dropout(dropout)
        
        # Feed-forward network with residual
        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim * 2),
            nn.Dropout(dropout)
        )
        self.ffn_norm = nn.LayerNorm(hidden_dim * 2)
        
        # Output layer
        self.hidden2tag = nn.Linear(hidden_dim * 2, num_labels)  # *2 for bidirectional
        
    def forward(self, x, mask=None, char_ids=None):
        # x: [batch_size, seq_len]
        word_emb = self.embedding(x)  # [batch_size, seq_len, embedding_dim]
        
        # Add character-level embeddings
        if self.use_char_emb and char_ids is not None:
            char_emb = self.char_cnn(char_ids)  # [batch_size, seq_len, 50]
            word_emb = torch.cat([word_emb, char_emb], dim=-1)  # [batch_size, seq_len, embedding_dim+50]
        
        # Layer normalization
        word_emb = self.embed_norm(word_emb)
        word_emb = self.dropout(word_emb)
        
        # LSTM
        lstm_out, _ = self.lstm(word_emb)  # [batch_size, seq_len, hidden_dim*2]
        lstm_out = self.dropout(lstm_out)
        
        # Multi-head attention
        if self.use_attention:
            lstm_out = self.attention(lstm_out, mask)
            lstm_out = self.dropout(lstm_out)
        
        # Feed-forward with residual
        residual = lstm_out
        ffn_out = self.ffn(lstm_out)
        lstm_out = self.ffn_norm(residual + ffn_out)  # Residual connection
        lstm_out = self.dropout(lstm_out)
        
        # Output layer
        logits = self.hidden2tag(lstm_out)  # [batch_size, seq_len, num_labels]
        return logits

## Step 4: Create Training Utilities

In [ ]:
%%writefile train_utils_scratch.py
"""Training utilities for from-scratch model."""

import json
import torch
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
from tqdm import tqdm
import numpy as np


def load_new_dataset(json_file_path):
    """Load dataset in the new format."""
    with open(json_file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    training_data = []
    for entry in data:
        if 'text' not in entry or 'annotations' not in entry:
            continue
        
        text = entry['text']
        annotations = entry['annotations']
        
        entities = []
        for ann in annotations:
            if not isinstance(ann, list) or len(ann) != 3:
                continue
            
            start, end, label = ann
            if start < 0 or end > len(text) or start >= end:
                continue
            
            entities.append((start, end, label))
        
        if entities:
            training_data.append((text, {"entities": entities}))
    
    return training_data


def build_vocab(data, min_freq=2):
    """Build vocabulary from training data."""
    word_freq = defaultdict(int)
    
    for text, _ in data:
        words = text.lower().split()
        for word in words:
            word_freq[word] += 1
    
    vocab = {'<PAD>': 0, '<UNK>': 1}
    idx = 2
    
    for word, freq in sorted(word_freq.items(), key=lambda x: x[1], reverse=True):
        if freq >= min_freq:
            vocab[word] = idx
            idx += 1
    
    return vocab


def create_tag_mappings(data):
    """Create tag mappings with BIO format."""
    all_labels = set(['O'])
    
    for text, entities in data:
        for start, end, label in entities['entities']:
            all_labels.add(f'B-{label}')
            all_labels.add(f'I-{label}')
    
    tags = sorted(list(all_labels))
    tag2idx = {tag: idx for idx, tag in enumerate(tags)}
    idx2tag = {idx: tag for tag, idx in tag2idx.items()}
    
    return tag2idx, idx2tag


def text_to_indices(text, vocab, max_len=500):
    """Convert text to sequence of word indices."""
    words = text.lower().split()
    indices = [vocab.get(word, vocab['<UNK>']) for word in words]
    
    if len(indices) > max_len:
        indices = indices[:max_len]
    else:
        indices = indices + [vocab['<PAD>']] * (max_len - len(indices))
    
    return indices


def align_labels_with_words(text, entities, max_len=500):
    """Align entity labels with word positions."""
    words = text.lower().split()
    labels = ['O'] * len(words)
    
    # Map character positions to word positions
    char_to_word = {}
    char_pos = 0
    for word_idx, word in enumerate(words):
        for _ in range(len(word)):
            char_to_word[char_pos] = word_idx
            char_pos += 1
        char_pos += 1  # space
    
    # Assign labels
    for start, end, label in sorted(entities, key=lambda x: x[0]):
        if start in char_to_word and end-1 in char_to_word:
            start_word = char_to_word[start]
            end_word = char_to_word[end-1]
            for i in range(start_word, end_word + 1):
                if i < len(labels):
                    if i == start_word:
                        labels[i] = f'B-{label}'
                    else:
                        labels[i] = f'I-{label}'
    
    if len(labels) > max_len:
        labels = labels[:max_len]
    else:
        labels = labels + ['O'] * (max_len - len(labels))
    
    return labels


class ResumeDatasetScratch(Dataset):
    """Dataset for from-scratch training."""
    def __init__(self, data, vocab, tag2idx, max_len=500):
        self.data = data
        self.vocab = vocab
        self.tag2idx = tag2idx
        self.max_len = max_len
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        text, entities = self.data[idx]
        
        input_ids = text_to_indices(text, self.vocab, self.max_len)
        labels = align_labels_with_words(text, entities['entities'], self.max_len)
        label_ids = [self.tag2idx.get(label, self.tag2idx['O']) for label in labels]
        
        # Create attention mask (1 for real tokens, 0 for padding)
        words = text.lower().split()
        actual_len = min(len(words), self.max_len)
        attention_mask = [1] * actual_len + [0] * (self.max_len - actual_len)
        
        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'labels': torch.tensor(label_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
        }

## Step 5: Load Data and Build Vocabulary

In [ ]:
import torch
from train_utils_scratch import load_new_dataset, build_vocab, create_tag_mappings, align_labels_with_words

# Load dataset
dataset_path = DRIVE_DATASET_PATH
print(f"Loading dataset from: {dataset_path}")
data = load_new_dataset(dataset_path)
print(f"✅ Loaded {len(data)} entries")

# Build vocabulary FROM YOUR DATA (not pre-trained)
print("\nBuilding vocabulary from training data...")
vocab = build_vocab(data, min_freq=2)
print(f"✅ Vocabulary size: {len(vocab)} (built from your data)")

# Create tag mappings
print("\nCreating tag mappings...")
tag2idx, idx2tag = create_tag_mappings(data)
print(f"✅ Number of labels: {len(tag2idx)}")
print(f"Sample labels: {list(tag2idx.keys())[:10]}")

# Split data
TRAIN_SPLIT = 0.9
split_idx = int(len(data) * TRAIN_SPLIT)
train_data = data[:split_idx]
val_data = data[split_idx:]
print(f"\n✅ Train: {len(train_data)} entries")
print(f"✅ Validation: {len(val_data)} entries")

## Step 6: Initialize Model (FROM SCRATCH) - OPTIMIZED VERSION

**Key Optimizations Applied:**
1. **Larger Architecture**: 128 embedding dim (↑ from 100), 384 hidden dim (↑ from 256), 3 layers (↑ from 2)
2. **Learning Rate Warmup**: Gradually increases LR in first 3 epochs for stable training
3. **Weight Decay**: L2 regularization (1e-5) prevents overfitting
4. **Gradient Clipping**: Prevents exploding gradients during training
5. **Longer Training**: 50 epochs with patience=10 for better convergence
6. **Balanced Dropout**: 0.5 (was 0.6) - better balance between regularization and learning

**Expected Improvements:**
- Better accuracy (larger model capacity)
- More stable training (warmup + gradient clipping)
- Less overfitting (weight decay + balanced dropout)
- Better convergence (longer training with patience)

## Step 6: Initialize Model (FROM SCRATCH)

In [ ]:
from model_from_scratch import BiLSTM_CRF_NER
from train_utils_scratch import ResumeDatasetScratch
from torch.utils.data import DataLoader

# Model hyperparameters (ADVANCED - Optimized for maximum accuracy)
EMBEDDING_DIM = 200  # Word embedding dimension (larger for richer representations)
HIDDEN_DIM = 512     # LSTM hidden dimension (larger capacity)
NUM_LAYERS = 4       # Number of LSTM layers (deeper network)
DROPOUT = 0.4        # Dropout rate (balanced for larger model)
MAX_LEN = 500        # Maximum sequence length
BATCH_SIZE = 16      # Batch size
EPOCHS = 60          # Number of epochs (more training)
LEARNING_RATE = 0.0008  # Learning rate (slightly lower for stability)
USE_CLASS_WEIGHTING = True  # Enable class weighting for imbalanced data
WEIGHT_DECAY = 1e-5  # L2 regularization (prevents overfitting)
GRADIENT_CLIP = 1.0  # Gradient clipping (prevents exploding gradients)
USE_WARMUP = True    # Learning rate warmup (helps early training)
WARMUP_EPOCHS = 5    # Number of warmup epochs (more for larger model)
USE_CHAR_EMB = True  # Use character-level embeddings (better OOV handling)
USE_ATTENTION = True # Use multi-head attention (better context understanding)

# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("="*60)
print("SYSTEM STATUS")
print("="*60)
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Using CPU - training will be slower")
print("="*60)

# Initialize model FROM SCRATCH (random weights)
print("\nInitializing model FROM SCRATCH...")
print("⚠️ No pre-trained models used - all weights are random!")
model = BiLSTM_CRF_NER(
    vocab_size=len(vocab),
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_labels=len(tag2idx),
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    use_char_emb=USE_CHAR_EMB,
    use_attention=USE_ATTENTION
)
model.to(device)

num_params = sum(p.numel() for p in model.parameters())
print(f"✅ Model initialized with {num_params:,} parameters")
print(f"✅ All weights are RANDOM (no pre-training)")

# Create datasets
train_dataset = ResumeDatasetScratch(train_data, vocab, tag2idx, MAX_LEN)
val_dataset = ResumeDatasetScratch(val_data, vocab, tag2idx, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

print(f"\n✅ Training batches: {len(train_loader)}")
print(f"✅ Validation batches: {len(val_loader)}")

## Step 7: Train the Model

In [ ]:
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from collections import Counter

# Calculate class weights for imbalanced data
if USE_CLASS_WEIGHTING:
    print("Calculating class weights for imbalanced data...")
    label_counts = Counter()
    
    for text, entities in train_data:
        labels = align_labels_with_words(text, entities['entities'], MAX_LEN)
        for label in labels:
            label_counts[label] += 1
    
    # Calculate weights (inverse frequency)
    total_labels = sum(label_counts.values())
    class_weights = torch.ones(len(tag2idx), device=device)
    
    for label, count in label_counts.items():
        if label in tag2idx and count > 0:
            # Weight = total / (num_classes * count)
            # This gives higher weight to rare classes
            weight = total_labels / (len(tag2idx) * count)
            class_weights[tag2idx[label]] = weight
    
    print(f"✅ Class weights calculated")
    print(f"   O label weight: {class_weights[tag2idx['O']]:.3f}")
    # Show top 5 rarest entities
    rarest = sorted(label_counts.items(), key=lambda x: x[1])[:5]
    for label, count in rarest:
        if label in tag2idx:
            print(f"   {label} weight: {class_weights[tag2idx[label]]:.3f} (count: {count})")
else:
    class_weights = None

# Loss and optimizer with class weighting
if USE_CLASS_WEIGHTING and class_weights is not None:
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    print("✅ Using weighted CrossEntropyLoss (helps with class imbalance)")
else:
    criterion = nn.CrossEntropyLoss()
    print("✅ Using standard CrossEntropyLoss")

# Optimizer with weight decay (L2 regularization)
optimizer = optim.Adam(
    model.parameters(), 
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY  # L2 regularization
)
print(f"✅ Optimizer: Adam with weight_decay={WEIGHT_DECAY}")

# Learning rate scheduler (reduce LR when validation loss plateaus)
# Note: verbose parameter not available in older PyTorch versions
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)
print("✅ Learning rate scheduler enabled (reduces LR on plateau)")
print("   Will reduce LR by 50% if no improvement for 3 epochs")

def train_epoch(model, dataloader, optimizer, criterion, device, tag2idx):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch in tqdm(dataloader, desc="Training"):
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        
        logits = model(input_ids)
        logits = logits.view(-1, logits.size(-1))
        labels_flat = labels.view(-1)
        mask_flat = attention_mask.view(-1)
        
        # Calculate loss (ignore padding tokens, not 'O' labels)
        # 'O' is a valid label, we only want to ignore actual padding
        active_loss = mask_flat == 1
        active_logits = logits[active_loss]
        active_labels = labels_flat[active_loss]
        
        if active_labels.numel() > 0:
            loss = criterion(active_logits, active_labels)
        else:
            loss = torch.tensor(0.0, device=device)
        
        optimizer.zero_grad()
        loss.backward()
        # Gradient clipping (prevents exploding gradients)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRADIENT_CLIP)
        optimizer.step()
        
        total_loss += loss.item()
        
        # Calculate accuracy (only on non-padding tokens, excluding 'O' for entity accuracy)
        predictions = torch.argmax(active_logits, dim=-1) if active_logits.numel() > 0 else torch.tensor([], device=device, dtype=torch.long)
        # For NER, we typically care about entity accuracy (not 'O' labels)
        entity_mask = (active_labels != tag2idx['O'])
        correct += ((predictions == active_labels) & entity_mask).sum().item()
        total += entity_mask.sum().item()
    
    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total if total > 0 else 0
    return avg_loss, accuracy


def validate(model, dataloader, criterion, device, tag2idx):
    """Validate the model."""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validation"):
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            
            logits = model(input_ids)
            logits = logits.view(-1, logits.size(-1))
            labels_flat = labels.view(-1)
            mask_flat = attention_mask.view(-1)
            
            # Calculate loss (ignore padding tokens)
            active_loss = mask_flat == 1
            active_logits = logits[active_loss]
            active_labels = labels_flat[active_loss]
            
            if active_labels.numel() > 0:
                loss = criterion(active_logits, active_labels)
            else:
                loss = torch.tensor(0.0, device=device)
            
            total_loss += loss.item()
            
            # Calculate accuracy (only on non-padding tokens, excluding 'O' for entity accuracy)
            predictions = torch.argmax(active_logits, dim=-1) if active_logits.numel() > 0 else torch.tensor([], device=device, dtype=torch.long)
            entity_mask = (active_labels != tag2idx['O'])
            correct += ((predictions == active_labels) & entity_mask).sum().item()
            total += entity_mask.sum().item()
    
    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total if total > 0 else 0
    return avg_loss, accuracy


# Training loop with early stopping
print(f"\n{'='*60}")
print("STARTING TRAINING FROM SCRATCH")
print(f"{'='*60}")
print(f"Epochs: {EPOCHS}")
print(f"Learning Rate: {LEARNING_RATE}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"{'='*60}\n")

best_val_loss = float('inf')
patience = 12  # Early stopping patience (increased for longer training with larger model)
patience_counter = 0

# Track training history
train_losses = []
val_losses = []
train_accs = []
val_accs = []

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")
    print("-" * 60)
    
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device, tag2idx)
    val_loss, val_acc = validate(model, val_loader, criterion, device, tag2idx)
    
    # Learning rate warmup (gradually increase LR in early epochs)
    if USE_WARMUP and epoch <= WARMUP_EPOCHS:
        warmup_lr = LEARNING_RATE * (epoch / WARMUP_EPOCHS)
        for param_group in optimizer.param_groups:
            param_group['lr'] = warmup_lr
        current_lr = warmup_lr
        print(f"🔥 Warmup epoch {epoch}/{WARMUP_EPOCHS} - LR: {current_lr:.6f}")
    else:
        # Update learning rate based on validation loss (after warmup)
        old_lr = optimizer.param_groups[0]['lr']
        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']
    
    print(f"✅ Train Loss: {train_loss:.4f}, Train Accuracy: {train_acc:.4f}")
    print(f"✅ Val Loss: {val_loss:.4f}, Val Accuracy: {val_acc:.4f}")
    
    if not (USE_WARMUP and epoch <= WARMUP_EPOCHS):
        print(f"📊 Learning Rate: {current_lr:.6f}", end="")
        # Manually log LR reduction (since verbose isn't available)
        if current_lr < old_lr:
            print(f" ⬇️  (Reduced from {old_lr:.6f})")
        else:
            print()
    
    # Track history
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save({
            'model_state_dict': model.state_dict(),
            'vocab': vocab,
            'tag2idx': tag2idx,
            'idx2tag': idx2tag,
            'embedding_dim': EMBEDDING_DIM,
            'hidden_dim': HIDDEN_DIM,
        }, '/content/model_from_scratch.bin')
        print(f"💾 Saved best model (val_loss: {val_loss:.4f}, val_acc: {val_acc:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\n⚠️ Early stopping triggered! No improvement for {patience} epochs.")
            print(f"Best validation loss: {best_val_loss:.4f}")
            print(f"Best validation accuracy: {max(val_accs):.4f}")
            break

print(f"\n{'='*60}")
print("TRAINING COMPLETE!")
print(f"{'='*60}")
print("✅ Model trained FROM SCRATCH (no pre-trained models used)")
print("✅ Model saved to: /content/model_from_scratch.bin")
print(f"{'='*60}")

## Step 8: Download Model

In [ ]:
# Download model to your computer
from google.colab import files
files.download('/content/model_from_scratch.bin')
print("✅ Model downloaded!")